In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("berkaykocaoglu/tr-sign-language")

print("Path to dataset files:", path)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2)
mp_drawing = mp.solutions.drawing_utils

DATASET_PATH = "/root/.cache/kagglehub/datasets/berkaykocaoglu/tr-sign-language/versions/1/tr_signLanguage_dataset/train"

X = []
y = []

for label in sorted(os.listdir(DATASET_PATH)):
    label_path = os.path.join(DATASET_PATH, label)
    if not os.path.isdir(label_path):
        continue

    print(f"Processing label: {label}")

    for img_name in tqdm(os.listdir(label_path)):
        img_path = os.path.join(label_path, img_name)

        image = cv2.imread(img_path)
        if image is None:
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)

        if results.multi_hand_landmarks:
            landmarks_all = []


            for hand_landmarks in results.multi_hand_landmarks:
                for lm in hand_landmarks.landmark:
                    landmarks_all.extend([lm.x, lm.y, lm.z])


            while len(landmarks_all) < 126:
                landmarks_all.extend([0.0, 0.0, 0.0])

            X.append(landmarks_all)
            y.append(label)


X = np.array(X)
y = np.array(y)

df = pd.DataFrame(X)
df['label'] = y
df.to_csv('hand_landmarks_dataset_double.csv', index=False)

print("✅ Çift el verisi oluşturuldu: hand_landmarks_dataset_double.csv")
